# Notebook 03 — Comprehend NER Inference

This notebook demonstrates how the pipeline runs NER inference using
Amazon Comprehend and processes the results.

**What this covers:**
- Mock Comprehend API responses (no real AWS calls needed)
- Structure of the entity dict returned by the wrapper
- Running real-time inference via `run_realtime()`
- Understanding `BeginOffset` / `EndOffset` for context expansion

**Why use a custom Comprehend endpoint:** The default Comprehend NER does not
know about sanctions-specific entity types.  A custom-trained model with
`SANCTIONED_PERSON` and `SANCTIONED_ORG` types gives us much higher precision
and recall for this specific task.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from unittest.mock import patch, MagicMock
import pandas as pd

## 1. Comprehend Entity Dict Schema

Every entity returned by `detect_entities_realtime()` has this structure:

```python
{
    "Text":        str,   # surface text of the entity
    "Type":        str,   # e.g. "SANCTIONED_PERSON"
    "Score":       float, # Comprehend confidence 0.0–1.0
    "BeginOffset": int,   # start character position in the input text
    "EndOffset":   int,   # character position AFTER the last character
}
```

`BeginOffset` and `EndOffset` use the **normalized** text character positions,
which is why we preserve the original text alongside `text_normalized`.

In [ ]:
# Mock Comprehend responses for three sentences.
mock_responses = {
    "JOHN DOE was added to the sanctions list.": [
        {
            "Text": "JOHN DOE",
            "Type": "SANCTIONED_PERSON",
            "Score": 0.98,
            "BeginOffset": 0,
            "EndOffset": 8,
        }
    ],
    "The vessel DARK STAR was seized last week.": [
        {
            "Text": "DARK STAR",
            "Type": "SANCTIONED_ORG",
            "Score": 0.91,
            "BeginOffset": 11,
            "EndOffset": 20,
        }
    ],
    "Muller GmbH was identified as a front company.": [
        {
            "Text": "Muller GmbH",
            "Type": "SANCTIONED_ORG",
            "Score": 0.72,
            "BeginOffset": 0,
            "EndOffset": 11,
        }
    ],
}

print("Mock responses defined for", len(mock_responses), "sentences.")

## 2. Entity Extraction via `detect_entities_realtime()`

We mock `boto3.client` so the wrapper calls our mock responses instead of
making real API requests.  The wrapper normalizes the Comprehend response
into the canonical entity dict schema.

In [ ]:
def make_mock_comprehend(responses_by_text):
    """Create a mock boto3 comprehend client that returns canned responses."""
    mock_client = MagicMock()
    def mock_detect(Text, EndpointArn):
        entities = responses_by_text.get(Text, [])
        return {"Entities": entities}
    mock_client.detect_entities.side_effect = mock_detect
    return mock_client

mock_client = make_mock_comprehend(mock_responses)

# Test single-sentence inference.
with patch('boto3.client', return_value=mock_client):
    from ner_ofac.comprehend_client import detect_entities_realtime
    sentence = "JOHN DOE was added to the sanctions list."
    entities = detect_entities_realtime(sentence, endpoint_arn="arn:fake")

print("Detected entities:")
for e in entities:
    print(f"  {e}")

## 3. Batch Inference via `run_realtime()`

`run_realtime()` iterates over a DataFrame of sentences and calls
`detect_entities_realtime()` for each row.  It returns a list of result
dicts, one per row, containing the row index, text, and detected entities.

In [ ]:
sentences_df = pd.DataFrame({
    'row_index': [0, 1, 2],
    'text': list(mock_responses.keys()),
    'text_normalized': list(mock_responses.keys()),
})

sentences_df

In [ ]:
with patch('boto3.client', return_value=mock_client):
    from ner_ofac.analysis.run_inference import run_realtime
    results = run_realtime(sentences_df, endpoint_arn="arn:fake")

print(f"Processed {len(results)} rows.\n")
for r in results:
    print(f"Row {r['row_index']}: {r['text'][:50]}")
    for e in r['entities']:
        print(f"  → [{e['Type']}] '{e['Text']}' (score={e['Score']:.2f}, offset={e['BeginOffset']}-{e['EndOffset']})")
    print()

## 4. Understanding Character Offsets

Character offsets are used by `context_expander.py` to locate the entity
within its sentence and expand the window for re-scoring REVIEW-tier matches.

In [ ]:
sentence = "JOHN DOE was added to the sanctions list."
begin, end = 0, 8

entity_span = sentence[begin:end]
context_before = sentence[max(0, begin-20):begin]
context_after = sentence[end:end+20]

print(f"Full sentence: {repr(sentence)}")
print(f"Entity span [{begin}:{end}]: {repr(entity_span)}")
print(f"Context before: {repr(context_before)}")
print(f"Context after: {repr(context_after)}")

## Summary

- Each detected entity has `Text`, `Type`, `Score`, `BeginOffset`, `EndOffset`
- `run_realtime()` processes a DataFrame and returns structured results
- Character offsets enable context expansion for REVIEW-tier matches
- The mock pattern here works without any real AWS credentials